# Minimum information tiers: paired omics

How many of the gene-metabolite associations of the MEANtools pilot (tomato) reach each tier, and what is missing in the graphs to reach the others?

**Case:** case 6, paired omics: transcript-metabolite associations (with cases 5 and 7). **Data:** association, annotation (evidence graphs of the pilots, `shape_constraints/rdf/` and `examples/`). **Queries:** the files of [`sparql-examples-pmw`](https://github.com/pathway-lod/sparql-examples-pmw/tree/main/examples/MinimumInformationTiers) (<https://pathway-lod.github.io/sparql-examples-pmw/>), run in a local Oxigraph store, with PlantMetWiki, Rhea and UniProt called by `SERVICE` on the live endpoints.

The outputs are a **dated snapshot**: the date and the PlantMetWiki release are in the summary table at the end. Each query is run once with a time limit; a tier that did not finish is shown as *timeout*. The results of every tier are also saved as CSV in `results/`. To rerun, open the notebook with a kernel that has the packages of `requirements.txt` installed (see the README of this folder).

In [1]:
import mi
import pandas as pd
from IPython.display import display
pd.set_option('display.max_colwidth', 80)
CASE = '6 paired omics'
print('PlantMetWiki release:', mi.pmw_release())

PlantMetWiki release: r17.0.0_20260605-171052


## The data

In [2]:
data = ['association', 'annotation']
print('associations:', mi.local_count(data, 'PREFIX lodp: <https://example.org/lod-pathways/pilot/> SELECT (COUNT(?a) AS ?n) WHERE { ?a a lodp:GeneMetaboliteAssociation }'))
print('distinct genes:', mi.local_count(data, 'PREFIX lodp: <https://example.org/lod-pathways/pilot/> PREFIX sio: <http://semanticscience.org/resource/> SELECT (COUNT(DISTINCT ?g) AS ?n) WHERE { ?a a lodp:GeneMetaboliteAssociation ; sio:SIO_000628 ?g . FILTER(STRSTARTS(STR(?g), "https://plants.ensembl.org/")) }'))
print('functional clusters:', mi.local_count(data, 'PREFIX lodp: <https://example.org/lod-pathways/pilot/> PREFIX sio: <http://semanticscience.org/resource/> SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { ?a a lodp:GeneMetaboliteAssociation ; sio:SIO_000772/lodp:functionalCluster ?c }'))

associations: 1293
distinct genes: 350


functional clusters: 52


## Gene and metabolite match the same pathway

`gene-and-metabolite-in-same-pathway`

gene-and-metabolite-in-same-pathway: An association of a gene and a metabolite feature where the gene (by its Solyc locus, listed in PlantMetWiki) and the metabolite (by an InChIKey first-block match of an annotation of the feature) are in the same PlantMetWiki pathway. Both sides are computed here; the stored anchor terms are not used. NOTE: the associations of the MEANtools pilot (MTBLS1039) have m/z values only and no annotation in the RDF, so the metabolite side returns no rows on the current graphs; `gene-in-plantmetwiki-pathway` shows the gene side alone. OUTPUT: a list of genes (supported members of a pathway: a gene associated with a metabolite that is in the same pathway, matched by Solyc locus and InChIKey); the number of associations and of metabolite features that support each gene-pathway pair are metadata.

*Keywords:* InChIKey, MEANtools, Minimum Information: evidence tiers, PlantMetWiki pathway, Solyc locus, case 5: metabolomics, metabolite annotations, case 6: paired omics, transcript-metabolite associations, evidence tier, federated query, gene, gene-metabolite association, local evidence graph, metabolite annotation, output: genes, paired omics

In [3]:
r = mi.run('gene-and-metabolite-in-same-pathway', ['association', 'annotation'], CASE, timeout=150)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

no rows | 0 rows | 2.1 s


,component,componentType,pathway,supportingAssociations,supportingMetaboliteFeatures


**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Gene side: the gene of an association is in a PlantMetWiki pathway

`gene-in-plantmetwiki-pathway`

gene-in-plantmetwiki-pathway: Gene side of `gene-and-metabolite-in-same-pathway`. The gene of each association is looked up in PlantMetWiki by its Solyc locus and the pathways that list it are returned. Counts the associations whose gene is in at least one pathway. OUTPUT: a list of genes (supported members of a pathway: the gene of an association is listed in a PlantMetWiki pathway by its Solyc locus), with the number of associations that support each gene-pathway pair as metadata.

*Keywords:* MEANtools, Minimum Information: evidence tiers, PlantMetWiki pathway, Solyc locus, case 6: paired omics, transcript-metabolite associations, evidence tier, federated query, gene, gene-metabolite association, local evidence graph, output: genes, paired omics

In [4]:
r = mi.run('gene-in-plantmetwiki-pathway', ['association'], CASE, timeout=240)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

no rows | 0 rows | 1.8 s


,component,componentType,pathway,supportingAssociations


**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Gene is an ortholog of a gene of a species that has the pathway

`ortholog-in-pathway-species`

ortholog-in-pathway-species: The metabolite matches a PlantMetWiki pathway (as in `gene-and-metabolite-in-same-pathway`) and the gene is an ortholog of a gene that PlantMetWiki lists in that pathway in another species. The orthology is read from the graph as lodp:orthologOf (for example from OrthoDB groups, see tier 5 of the pathway-component queries); the gene of the other species is matched to the pathway by its UniProt accession. NOT RUNNABLE ON THE CURRENT GRAPHS: orthologs are not in the RDF yet. OUTPUT: a list of genes (putative members of a pathway: the gene is an ortholog of a gene listed in that pathway in another species, and its associated metabolite is in the pathway); the ortholog and the number of supporting associations are metadata.

*Keywords:* InChIKey, Minimum Information: evidence tiers, OrthoDB, PlantMetWiki pathway, UniProt, case 6: paired omics, transcript-metabolite associations, evidence tier, federated query, gene-metabolite association, local evidence graph, metabolite annotation, no data yet, orthologs, output: genes, paired omics

In [5]:
r = mi.run('ortholog-in-pathway-species', ['association', 'annotation'], CASE, timeout=150)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

no rows | 0 rows | 4.4 s


,component,componentType,pathway,orthologUniprot,supportingAssociations


**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Gene is predicted to catalyse the transformation between the two metabolites

`gene-catalyses-the-transformation`

gene-catalyses-the-transformation: A gene is associated with two metabolite features. Both are matched to PlantMetWiki metabolites (`annotation-matches-plantmetwiki`), and PlantMetWiki has a conversion from one to the other (substrate to product) catalysed by an enzyme that the gene is linked to (here by its UniProt accession, lodp:uniprot). Returns the gene, the two metabolites and the conversion. NOT RUNNABLE ON THE CURRENT GRAPHS: the associations have no second metabolite per gene with an annotation, and no gene-to-UniProt link in the RDF. OUTPUT: a list of reactions (putative or supported: the PlantMetWiki conversion between two metabolites, catalysed by an enzyme that a co-expressed gene encodes) with their pathway; the gene, the enzyme (UniProt) and the two associated metabolite features (substrate and product) are metadata.

*Keywords:* InChIKey, Minimum Information: evidence tiers, PlantMetWiki reaction, UniProt, case 2: inferred enzyme function from PMN/PlantCyc+ db, case 5: metabolomics, metabolite annotations, case 6: paired omics, transcript-metabolite associations, enzyme, evidence tier, federated query, gene-metabolite association, local evidence graph, metabolite annotation, no data yet, output: reactions, paired omics, substrate and product

In [6]:
r = mi.run('gene-catalyses-the-transformation', ['association', 'annotation'], CASE, timeout=150)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

no rows | 0 rows | 0.0 s


,component,componentType,pathway,gene,substrateFeature,productFeature,uniprot


**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Ortholog in a pathway species and a machine-learning predicted transformation

`ortholog-plus-ml-prediction`

ortholog-plus-ml-prediction: As in `ortholog-in-pathway-species`, and the transformation also matches an enzymatic transformation predicted by a model on the PlantMetWiki graph (case 7, for example the predictions of PlantMetBench): a predicted link between the enzyme the gene is an ortholog of and a reaction that connects the two metabolites. The prediction is read as lodp:PredictedLink with lodp:predictionScore and the model run. NOT RUNNABLE ON THE CURRENT GRAPHS: no predictions are in the RDF yet. OUTPUT: a list of reactions (putative: a reaction predicted by a machine-learning model for an enzyme that the gene is an ortholog of), with the pathway when it is known; the gene, the enzyme, the model run and the prediction score are metadata.

*Keywords:* Minimum Information: evidence tiers, case 6: paired omics, transcript-metabolite associations, case 7: unsupervised GNN predictions of reactions (conversion steps), evidence tier, gene-metabolite association, local evidence graph, machine learning prediction, no data yet, orthologs, output: reactions, paired omics, predicted enzymatic transformation

In [7]:
r = mi.run('ortholog-plus-ml-prediction', ['association', 'annotation'], CASE, timeout=150)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

no rows | 0 rows | 0.0 s


,component,componentType,pathway,gene,enzyme,score,run


**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Summary of this case

In [8]:
s = pd.read_csv('results/summary.csv')
display(s[s['case'] == CASE].reset_index(drop=True))

,tier,case,status,n_rows,seconds,timeout_s,date,pmw_release
0,gene-and-metabolite-in-same-pathway,6 paired omics,no rows,0,2.1,150,2026-11-11,r17.0.0_20260605-171052
1,gene-catalyses-the-transformation,6 paired omics,no rows,0,0.0,150,2026-11-11,r17.0.0_20260605-171052
2,gene-in-plantmetwiki-pathway,6 paired omics,no rows,0,1.8,240,2026-11-11,r17.0.0_20260605-171052
3,ortholog-in-pathway-species,6 paired omics,no rows,0,4.4,150,2026-11-11,r17.0.0_20260605-171052
4,ortholog-plus-ml-prediction,6 paired omics,no rows,0,0.0,150,2026-11-11,r17.0.0_20260605-171052
